# ChatML

Goal: executar os experimentos centrais do tutorial em TypeScript com o kernel Deno.

Tutorial: `tutorials/01-chat-ml.md`


## Setup do ambiente

Este notebook implementa os exemplos do tutorial usando chamadas reais para a API legado de `completions` e o SDK oficial para o exemplo moderno.


In [ ]:
import OpenAI from "npm:openai";

const apiKey = Deno.env.get("OPENAI_API_KEY");
const legacyModel = "text-davinci-002";
const client = apiKey ? new OpenAI({ apiKey }) : null;

console.log(apiKey ? "OPENAI_API_KEY carregada" : "Defina OPENAI_API_KEY no .env e inicie com npm run notebook ou npm run lab");
console.log("Modelo legado configurado:", legacyModel);


In [ ]:
async function callLegacyCompletion({ prompt, max_tokens = 120, temperature = 0.7, stop }) {
  if (!apiKey) {
    throw new Error("Defina OPENAI_API_KEY no .env antes de executar.");
  }

  const response = await fetch("https://api.openai.com/v1/completions", {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      Authorization: `Bearer ${apiKey}`
    },
    body: JSON.stringify({
      model: legacyModel,
      prompt,
      max_tokens,
      temperature,
      ...(stop ? { stop } : {})
    })
  });

  const data = await response.json();

  if (!response.ok) {
    console.error(data);
    throw new Error(`Legacy completions request failed: ${response.status}`);
  }

  return data;
}


## Parte 1: `completions` continua a sequencia

Corresponde ao experimento do tutorial que envia apenas um texto simples e observa a continuacao gerada.


In [ ]:
const helloResponse = await callLegacyCompletion({
  prompt: "Ola",
  max_tokens: 20,
  temperature: 0.7
});

console.log(helloResponse.choices[0].text);
console.log("Finish reason:", helloResponse.choices[0].finish_reason);


## Parte 2: formato de conversa via prompt estruturado

Aqui o objetivo e inspecionar como o modelo continua um prompt no formato `User:` / `Assistant:` sem controle de parada.


In [ ]:
const promptAsDialogue = `User: Ola, tudo bem?\nAssistant:\n`;

const conversationResponse = await callLegacyCompletion({
  prompt: promptAsDialogue,
  max_tokens: 150,
  temperature: 0.7
});

console.log(conversationResponse.choices[0].text);


## Parte 3: `stop` controla o encerramento do turno

O codigo abaixo executa o mesmo prompt anterior, agora com corte explicito antes do proximo turno do usuario.


In [ ]:
const stopControlledResponse = await callLegacyCompletion({
  prompt: promptAsDialogue,
  max_tokens: 150,
  temperature: 0.7,
  stop: ["User:"]
});

console.log(stopControlledResponse.choices[0].text);
console.log("Finish reason:", stopControlledResponse.choices[0].finish_reason);


## Parte 4: ChatML explicita papeis e fronteiras

Este bloco serializa as mensagens manualmente em ChatML e usa `stop` para encerrar no fim do bloco do assistant.


In [ ]:
function toChatML(messages) {
  return messages
    .map(({ role, content }) => `<|im_start|>${role}\n${content}\n<|im_end|>`)
    .join("\n");
}

const chatmlPrompt = `${toChatML([
  { role: "system", content: "Voce e um assistente util." },
  { role: "user", content: "Explique computacao em nuvem em 2 frases." }
])}\n<|im_start|>assistant\n`;

const chatmlResponse = await callLegacyCompletion({
  prompt: chatmlPrompt,
  max_tokens: 120,
  temperature: 0.7,
  stop: ["<|im_end|>"]
});

console.log(chatmlResponse.choices[0].text);
console.log("Finish reason:", chatmlResponse.choices[0].finish_reason);


## Parte 5: equivalente moderno com SDK

O tutorial fecha conectando o formato legado a APIs modernas. Aqui usamos o SDK oficial da OpenAI para mostrar essa diferenca de interface.


In [ ]:
if (!client) {
  throw new Error("Defina OPENAI_API_KEY no .env antes de executar.");
}

const modernResponse = await client.responses.create({
  model: "gpt-4.1-mini",
  input: [
    { role: "system", content: "Voce e um assistente util." },
    { role: "user", content: "Explique computacao em nuvem em 2 frases." }
  ]
});

console.log(modernResponse.output_text);
